In [4]:
import requests
import json

In [60]:
with open("./credentials.json", "r") as cred:
        credentials = json.load(cred)
        CLIENT_ID = credentials["redditClientID"]
        SECRET_KEY = credentials["redditSecretKey"]
        USERNAME = credentials["redditUsername"]
        PASS = credentials["redditPass"]
        USER_AGENT = credentials["redditUserAgent"]
        MONGO_CRED = credentials["mongoString"]

In [29]:
# Documentation : https://github.com/reddit-archive/reddit/wiki/OAuth2-Quick-Start-Example
client_auth = requests.auth.HTTPBasicAuth(f'{CLIENT_ID}', f'{SECRET_KEY}')
post_data = {"grant_type": "password", "username": f"{USERNAME}", "password": f"{PASS}"}
headers = {"User-Agent": f"{USER_AGENT}"}

In [30]:
try:
    response = requests.post("https://www.reddit.com/api/v1/access_token", auth=client_auth, data=post_data, headers=headers)
    #Save Token 
    token_type = response.json()['token_type']
    access_token = response.json()['access_token']
except:
    print(f"Error generating token, status code: {response.status_code}")

In [31]:
new_headers = {"Authorization": f"{token_type} {access_token}", **headers}
#Now send GET requests to "https://oauth.reddit.com"

In [48]:
def get_subreddit_data(subreddit):
    #Add testing as to whether that is an actual subreddit
    try:
        get_posts = requests.get(f"https://oauth.reddit.com/r/{subreddit}/new", headers=new_headers)
        if get_posts.status_code == 200:
            print(f"collected 25 posts from r/{subreddit}")
            return get_posts
        else:
            raise
    except:
        print(f"Error getting posts, status code: {get_posts.status_code}")

In [52]:
test = get_subreddit_data("indieheads")

collected 25 posts from r/indieheads


In [73]:
test.json()["data"]["children"]

[{'kind': 't3',
  'data': {'approved_at_utc': None,
   'subreddit': 'indieheads',
   'selftext': '',
   'author_fullname': 't2_o9eg5byp',
   'saved': False,
   'mod_reason_title': None,
   'gilded': 0,
   'clicked': False,
   'title': 'The New Stereogum Membership Program Is Here',
   'link_flair_richtext': [],
   'subreddit_name_prefixed': 'r/indieheads',
   'hidden': False,
   'pwls': 6,
   'link_flair_css_class': None,
   'downs': 0,
   'thumbnail_height': 78,
   'top_awarded_type': None,
   'hide_score': True,
   'name': 't3_vocc42',
   'quarantine': False,
   'link_flair_text_color': 'dark',
   'upvote_ratio': 0.86,
   'author_flair_background_color': None,
   'subreddit_type': 'public',
   'ups': 5,
   'total_awards_received': 0,
   'media_embed': {},
   'thumbnail_width': 140,
   'author_flair_template_id': None,
   'is_original_content': False,
   'user_reports': [],
   'secure_media': None,
   'is_reddit_media_domain': False,
   'is_meta': False,
   'category': None,
   'secur

In [61]:
def mongo_connection():
    return pymongo.MongoClient(MONGO_CRED)

In [62]:
def select_db(client, database):
    return client[database]

In [63]:
def select_collection(database, collection_name):
    return database[collection_name]

In [65]:
def load_posts(posts, collection):
    collection.insertMany(posts)